In [1]:
print("Let's start")

Let's start


In [2]:
import pandas as pd

In [3]:
df = pd.read_csv(r"C:\Users\Abhishek sharma\Artificial Intelligence\Creating Datasets\My Dataset\processed_imdb_dataset.csv")
df.head()

,Unnamed: 0,review,sentiment
0,0,one review mention watch 1 oz episod youll hoo...,positive
1,1,wonder littl product film techniqu unassum old...,positive
2,2,thought wonder way spend time hot summer weeke...,positive
3,3,basic there famili littl boy jake think there ...,negative
4,4,petter mattei love time money visual stun film...,positive


In [4]:
# Word2Vec training template (gensim)
import multiprocessing
import pandas as pd
import numpy as np
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords', quiet=True)

# 1) Load your dataset (adjust path/column)
df = pd.read_csv(r"C:\Users\Abhishek sharma\Artificial Intelligence\Creating Datasets\My Dataset\processed_imdb_dataset.csv")
texts = df['review'].astype(str).fillna('').tolist()   # change 'review' to your text column

# 2) Preprocessing / tokenization
stop_words = set(stopwords.words('english'))

def preprocess(text):
    # simple tokenization + lower + remove very short tokens and stopwords
    tokens = simple_preprocess(text, deacc=True, min_len=2)  # deacc=True removes punctuation
    tokens = [t for t in tokens if t not in stop_words]
    return tokens

sentences = [preprocess(t) for t in texts]

# Optional: inspect some tokenized sentences
print("Example tokens:", sentences[:3])

# 3) Train Word2Vec
workers = max(1, multiprocessing.cpu_count() - 1)
model = Word2Vec(
    sentences=sentences,
    vector_size=100,    # dimensionality
    window=5,           # context window
    min_count=5,        # ignore rare words
    workers=workers,
    sg=1,               # 1 = skip-gram; 0 = CBOW
    epochs=10,
    seed=42
)

# 4) Save the model
model.save("word2vec_imdb.model")
# or save vectors in word2vec text/binary format:
model.wv.save_word2vec_format("word2vec_imdb.vec", binary=False)

# 5) Useful examples
wv = model.wv
print("Vocab size:", len(wv.index_to_key))
print("Most similar to 'movie':", wv.most_similar("movie", topn=10))
print("Vector for 'movie' (len):", len(wv['movie']))

# 6) Create document vectors by averaging word vectors (simple)
def document_vector(tokens, model):
    # filter tokens that are in the model's vocabulary
    vecs = [model.wv[t] for t in tokens if t in model.wv]
    if len(vecs) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vecs, axis=0)

# Example: compute vectors for first 5 documents
doc_vectors = np.vstack([document_vector(s, model) for s in sentences[:5]])
print("Doc vectors shape:", doc_vectors.shape)

# 7) Downstream usage: clustering / classifier
# e.g., use `doc_vectors` as features for scikit-learn models

Example tokens: [['one', 'review', 'mention', 'watch', 'oz', 'episod', 'youll', 'hook', 'right', 'exactli', 'happen', 'first', 'thing', 'struck', 'oz', 'brutal', 'unflinch', 'scene', 'violenc', 'set', 'right', 'word', 'go', 'trust', 'show', 'faint', 'heart', 'timid', 'show', 'pull', 'punch', 'regard', 'drug', 'sex', 'violenc', 'hardcor', 'classic', 'use', 'word', 'call', 'oz', 'nicknam', 'given', 'oswald', 'maximum', 'secur', 'state', 'penitentari', 'focus', 'mainli', 'emerald', 'citi', 'experiment', 'section', 'prison', 'cell', 'glass', 'front', 'face', 'inward', 'privaci', 'high', 'agenda', 'em', 'citi', 'home', 'manyaryan', 'muslim', 'gangsta', 'latino', 'christian', 'italian', 'irish', 'moreso', 'scuffl', 'death', 'stare', 'dodgi', 'deal', 'shadi', 'agreement', 'never', 'far', 'away', 'would', 'say', 'main', 'appeal', 'show', 'due', 'fact', 'goe', 'show', 'wouldnt', 'dare', 'forget', 'pretti', 'pictur', 'paint', 'mainstream', 'audienc', 'forget', 'charm', 'forget', 'romanceoz', 'do